[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Luckeywolf/colab-torrent-to-drive/blob/main/Torrent_To_Google_Drive.ipynb)


# ⚡ Turbo Torrent Downloader to Google Drive (Technical Edition)
Hardware-tuned throughput, live tier-1 tracker synchronization, 16 MB buffered Drive sync, and monospace telemetry dashboard.

In [ ]:
# ==========================================
# ⚡ Turbo Torrent Downloader (Minimal & Technical Dashboard)
# ==========================================
from google.colab import drive
import os
import subprocess
import re
import shutil
import urllib.request
import time
from IPython.display import HTML, display

# 1. Mount Google Drive
print("Mounting Google Drive...")
drive.mount("/content/drive")

# 2. Instant Install (skips slow mirror list check)
print("Initializing Aria2...")
!apt-get install -y -qq aria2 > /dev/null
print("Aria2 ready.")

# 3. Setup paths (Local SSD cache + Google Drive destination)
LOCAL_CACHE_DIR = "/content/download_cache"
DRIVE_SAVE_DIR = "/content/drive/My Drive/Torrent"

os.makedirs(LOCAL_CACHE_DIR, exist_ok=True)
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)

# 4. Inject Live Public Trackers
print("Synchronizing BitTorrent trackers...")
trackers_arg = ""
try:
    tracker_url = "https://raw.githubusercontent.com/ngosang/trackerslist/master/trackers_best.txt"
    req = urllib.request.Request(tracker_url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=3) as resp:
        trackers = [line.strip().decode("utf-8") for line in resp if line.strip()]
        if trackers:
            trackers_arg = ",".join(trackers)
            print(f"Loaded {len(trackers)} Tier-1 trackers.")
except Exception:
    print("Using DHT network.")

# 5. Target Input
target = input("\nEnter Magnet link or .torrent URL (or press Enter to upload file): ").strip()

if not target:
    from google.colab import files
    print("Upload your .torrent file:")
    uploaded = files.upload()
    if uploaded:
        target = list(uploaded.keys())[0]

def render_progress_card(percent=0.0, speed="0 B", downloaded="0 B", total="0 B", eta="--", peers="0", status="INITIALIZING"):
    return f"""
    <div style="font-family: 'SF Mono', 'JetBrains Mono', 'Fira Code', Menlo, Consolas, monospace; background: #0c0d10; border: 1px solid #1e222b; border-radius: 8px; padding: 16px 18px; max-width: 580px; color: #d1d5db; box-shadow: 0 4px 20px rgba(0,0,0,0.5); font-size: 12px; margin: 8px 0;">
      <div style="display: flex; justify-content: space-between; align-items: center; padding-bottom: 10px; border-bottom: 1px solid #181b22; margin-bottom: 12px;">
        <div style="display: flex; align-items: center; gap: 8px; font-weight: 600; color: #f3f4f6; letter-spacing: 0.5px;">
          <span style="display: inline-block; width: 7px; height: 7px; border-radius: 50%; background: #10b981; box-shadow: 0 0 6px #10b981;"></span>
          <span>{status.upper()}</span>
        </div>
        <div style="color: #9ca3af; font-size: 11px;">
          ETA <span style="color: #38bdf8; font-weight: 600;">{eta}</span>
        </div>
      </div>

      <div style="display: grid; grid-template-columns: repeat(4, 1fr); gap: 8px; margin-bottom: 12px;">
        <div>
          <div style="color: #6b7280; font-size: 10px; text-transform: uppercase; letter-spacing: 0.5px;">RATE</div>
          <div style="color: #10b981; font-size: 14px; font-weight: 600; margin-top: 2px;">{speed}/s</div>
        </div>
        <div>
          <div style="color: #6b7280; font-size: 10px; text-transform: uppercase; letter-spacing: 0.5px;">TRANSFERRED</div>
          <div style="color: #e5e7eb; font-size: 13px; font-weight: 500; margin-top: 2px;">{downloaded}</div>
        </div>
        <div>
          <div style="color: #6b7280; font-size: 10px; text-transform: uppercase; letter-spacing: 0.5px;">TOTAL</div>
          <div style="color: #9ca3af; font-size: 13px; font-weight: 500; margin-top: 2px;">{total}</div>
        </div>
        <div>
          <div style="color: #6b7280; font-size: 10px; text-transform: uppercase; letter-spacing: 0.5px;">PEERS</div>
          <div style="color: #a78bfa; font-size: 14px; font-weight: 600; margin-top: 2px;">{peers}</div>
        </div>
      </div>

      <div style="background: #161922; border-radius: 3px; height: 6px; overflow: hidden; margin-bottom: 8px; border: 1px solid #232734;">
        <div style="background: #0ea5e9; height: 100%; width: {percent}%; transition: width 0.2s ease;"></div>
      </div>

      <div style="display: flex; justify-content: space-between; font-size: 11px; color: #6b7280;">
        <span>COMPLETION</span>
        <span style="color: #0ea5e9; font-weight: 600;">{percent:.2f}%</span>
      </div>
    </div>
    """

def render_syncing_card():
    return """
    <div style="font-family: 'SF Mono', 'JetBrains Mono', 'Fira Code', Menlo, Consolas, monospace; background: #0c0d10; border: 1px solid #2e2616; border-radius: 8px; padding: 14px 18px; max-width: 580px; color: #d1d5db; font-size: 12px; margin: 8px 0;">
      <div style="display: flex; align-items: center; gap: 8px; color: #f59e0b; font-weight: 600; margin-bottom: 4px;">
        <span>⟳</span>
        <span>SYNCING TO GOOGLE DRIVE</span>
      </div>
      <div style="color: #9ca3af; font-size: 11px;">Streaming verified blocks to Drive FUSE mount (16 MB buffer) & flushing...</div>
    </div>
    """

def render_completed_card(save_path, total_size_str):
    return f"""
    <div style="font-family: 'SF Mono', 'JetBrains Mono', 'Fira Code', Menlo, Consolas, monospace; background: #0c0d10; border: 1px solid #132e1e; border-radius: 8px; padding: 14px 18px; max-width: 580px; color: #d1d5db; font-size: 12px; margin: 8px 0;">
      <div style="display: flex; justify-content: space-between; align-items: center; margin-bottom: 6px;">
        <div style="display: flex; align-items: center; gap: 8px; color: #10b981; font-weight: 600;">
          <span>✓</span>
          <span>TRANSFER COMPLETE ({total_size_str})</span>
        </div>
        <div style="color: #6b7280; font-size: 10px;">FLUSHED & VERIFIED</div>
      </div>
      <div style="color: #9ca3af; font-size: 11px; font-family: monospace; word-break: break-all; background: #11141b; padding: 6px 10px; border-radius: 4px; border: 1px solid #1a202c; margin-top: 6px;">
        {save_path}
      </div>
    </div>
    """

def parse_aria2_progress(line):
    m_size = re.search(r"\[#\w+\s+([0-9.]+[KMGT]?i?B)/([0-9.]+[KMGT]?i?B)\(([0-9.]+)%\)\]", line)
    if not m_size:
        m_size = re.search(r"\[#\w+\s+([0-9.]+[KMGT]?i?B)/([0-9.]+[KMGT]?i?B)\(([0-9.]+)%\)", line)
    if not m_size:
        return None
    downloaded, total, percent = m_size.groups()
    m_cn = re.search(r"CN:(\d+)", line)
    cn = m_cn.group(1) if m_cn else "0"
    m_dl = re.search(r"DL:([0-9.]+[KMGT]?i?B)", line)
    dl = m_dl.group(1) if m_dl else "0B"
    m_eta = re.search(r"ETA:([0-9A-Za-z]+)", line)
    eta = m_eta.group(1) if m_eta else "--"
    return {
        "downloaded": downloaded,
        "total": total,
        "percent": float(percent),
        "peers": cn,
        "speed": dl,
        "eta": eta
    }

def fast_copy_file(src, dst, chunk_size=16 * 1024 * 1024):
    with open(src, "rb") as fsrc, open(dst, "wb") as fdst:
        while True:
            chunk = fsrc.read(chunk_size)
            if not chunk:
                break
            fdst.write(chunk)

if target:
    cmd = [
        "aria2c",
        "-c",
        "--enable-dht=true",
        "--bt-enable-lpd=true",
        "--max-connection-per-server=16",
        "--split=16",
        "--bt-max-peers=128",
        "--disk-cache=64M",
        "--file-allocation=falloc",
        "--bt-request-peer-speed-limit=200K",
        "--seed-time=0",
        "--summary-interval=1",
        f"--dir={LOCAL_CACHE_DIR}",
    ]
    if trackers_arg:
        cmd.append(f"--bt-tracker={trackers_arg}")
    cmd.append(target)

    display_handle = display(HTML(render_progress_card(status="CONNECTING")), display_id=True)
    
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )

    buffer = ""
    while True:
        chunk = proc.stdout.read(256)
        if not chunk and proc.poll() is not None:
            break
        buffer += chunk
        while "\r" in buffer or "\n" in buffer:
            idx_r = buffer.find("\r")
            idx_n = buffer.find("\n")
            if idx_r != -1 and (idx_n == -1 or idx_r < idx_n):
                line, buffer = buffer[:idx_r], buffer[idx_r + 1:]
            else:
                line, buffer = buffer[:idx_n], buffer[idx_n + 1:]
            line = line.strip()
            if line:
                data = parse_aria2_progress(line)
                if data:
                    display_handle.update(HTML(render_progress_card(
                        percent=data["percent"],
                        speed=data["speed"],
                        downloaded=data["downloaded"],
                        total=data["total"],
                        eta=data["eta"],
                        peers=data["peers"],
                        status="DOWNLOADING"
                    )))
                elif "downloading metadata" in line.lower():
                    display_handle.update(HTML(render_progress_card(status="METADATA")))

    if proc.returncode == 0:
        display_handle.update(HTML(render_syncing_card()))
        
        total_bytes = 0
        for item in os.listdir(LOCAL_CACHE_DIR):
            src = os.path.join(LOCAL_CACHE_DIR, item)
            dst = os.path.join(DRIVE_SAVE_DIR, item)
            if os.path.isdir(src):
                os.makedirs(dst, exist_ok=True)
                for root, dirs, files_list in os.walk(src):
                    rel = os.path.relpath(root, src)
                    target_root = os.path.join(dst, rel)
                    os.makedirs(target_root, exist_ok=True)
                    for f in files_list:
                        s_file = os.path.join(root, f)
                        d_file = os.path.join(target_root, f)
                        fast_copy_file(s_file, d_file)
                        total_bytes += os.path.getsize(s_file)
            else:
                fast_copy_file(src, dst)
                total_bytes += os.path.getsize(src)
        
        os.sync()
        
        size_mb = total_bytes / (1024 * 1024)
        size_str = f"{size_mb / 1024:.2f} GB" if size_mb >= 1024 else f"{size_mb:.1f} MB"
        
        display_handle.update(HTML(render_completed_card(DRIVE_SAVE_DIR, size_str)))
    else:
        print("Aria2 exited with error code:", proc.returncode)
else:
    print("No link or file provided.")

